# Machine Learning — Clustering & Classification du Sentiment
**Source** : BigQuery `gdelt-494812.benin_2025.events_clean`

Ce notebook produit :
1. Un clustering K-Means pour identifier des profils d'événements
2. Un classifieur Random Forest pour prédire le sentiment médiatique
3. Les métriques d'évaluation (silhouette, F1, cross-validation)
4. Les modèles sauvegardés dans `/models/`

## 0. Imports & connexion

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import joblib
from pathlib import Path
from google.cloud import bigquery

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    silhouette_score, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

PROJECT = 'gdelt-494812'
DATASET = 'benin_2025'
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

client = bigquery.Client(project=PROJECT)
def bq(sql): return client.query(sql).to_dataframe()

print('Prêt.')

AttributeError: module 'google.auth.credentials' has no attribute 'CredentialsWithRegionalAccessBoundary'

## 1. Chargement & préparation des features

In [ ]:
df_raw = bq(f"SELECT * FROM `{PROJECT}.{DATASET}.events_clean` LIMIT 100000")
print(f'Données : {len(df_raw):,} lignes — Colonnes : {list(df_raw.columns)}')

In [ ]:
def find_col(df, keywords):
    for kw in keywords:
        for c in df.columns:
            if kw.lower() in c.lower(): return c
    return None

COL_DATE      = find_col(df_raw, ['date', 'sqldate'])
COL_TONE      = find_col(df_raw, ['tone', 'avgtone', 'avg_tone'])
COL_GOLDSTEIN = find_col(df_raw, ['goldstein'])
COL_QUADCLASS = find_col(df_raw, ['quadclass', 'quad_class', 'category'])
COL_MENTIONS  = find_col(df_raw, ['nummentions', 'mentions'])
COL_SOURCES   = find_col(df_raw, ['numsources', 'sources'])
COL_ARTICLES  = find_col(df_raw, ['numarticles', 'articles'])
COL_EVENTCODE = find_col(df_raw, ['eventcode', 'event_code'])

print({k: v for k, v in [
    ('date', COL_DATE), ('tone', COL_TONE), ('goldstein', COL_GOLDSTEIN),
    ('quadclass', COL_QUADCLASS), ('mentions', COL_MENTIONS),
    ('sources', COL_SOURCES), ('articles', COL_ARTICLES)
]})

In [ ]:
df = df_raw.copy()

# Colonnes numériques de base — forcer la conversion
for col in [COL_TONE, COL_GOLDSTEIN, COL_QUADCLASS, COL_MENTIONS, COL_SOURCES, COL_ARTICLES]:
    if col: df[col] = pd.to_numeric(df[col], errors='coerce')

# Features temporelles
if COL_DATE:
    df['_date']  = pd.to_datetime(df[COL_DATE].astype(str), errors='coerce')
    df['_month'] = df['_date'].dt.month
    df['_dow']   = df['_date'].dt.dayofweek  # 0=lundi

# Feature binaire conflit
if COL_QUADCLASS:
    df['_is_conflict'] = (df[COL_QUADCLASS] >= 3).astype(int)

# Label sentiment (cible) — np.select évite NaN→"nan" de pd.cut
if COL_TONE:
    _tone = df[COL_TONE]
    df['_sentiment'] = np.select(
        [_tone < -2, _tone > 2],
        ['Negatif', 'Positif'],
        default='Neutre'
    )
    df.loc[_tone.isna(), '_sentiment'] = np.nan
    print('Distribution du sentiment :')
    print(df['_sentiment'].value_counts())

# Supprimer les lignes avec trop de NaN
# _sentiment est inclus pour éviter des classes NaN dans stratify
FEATURE_COLS = [c for c in [COL_TONE, COL_GOLDSTEIN, COL_QUADCLASS,
                              COL_MENTIONS, COL_SOURCES, COL_ARTICLES,
                              '_month', '_dow', '_is_conflict'] if c]
TARGET_COL = '_sentiment' if COL_TONE else None
drop_cols = FEATURE_COLS + ([TARGET_COL] if TARGET_COL else [])
df_ml = df.dropna(subset=drop_cols).copy()
print(f'\nDataset ML après nettoyage : {len(df_ml):,} lignes')


---
## 2. Clustering K-Means — Identifier des profils d'événements

In [ ]:
# Features pour le clustering (sans le ton — c'est ce qu'on veut prédire)
CLUSTER_FEATS = [c for c in [COL_GOLDSTEIN, COL_QUADCLASS,
                               COL_MENTIONS, COL_SOURCES, '_month', '_is_conflict'] if c]
X_clust = df_ml[CLUSTER_FEATS].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clust)

# Méthode du coude
inertias, sil_scores = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

from plotly.subplots import make_subplots
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Methode du coude (inertie)', 'Score de silhouette'])
fig.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
    name='Inertie', line=dict(color='#e91e8c')), row=1, col=1)
fig.add_trace(go.Scatter(x=list(K_range), y=sil_scores, mode='lines+markers',
    name='Silhouette', line=dict(color='#6c5ce7')), row=1, col=2)
fig.update_layout(title='Choix du nombre de clusters', height=350, showlegend=False)
fig.show()

best_k = K_range[np.argmax(sil_scores)]
print(f'K suggéré par silhouette : {best_k} (score = {max(sil_scores):.3f})')
print('Choisis le K qui correspond au coude ET au meilleur score de silhouette.')

In [ ]:
# Appliquer le K choisi — modifie si nécessaire
K_FINAL = best_k  # ou remplace par un entier ex: K_FINAL = 4

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df_ml['cluster'] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df_ml['cluster'])
print(f'Score de silhouette final : {sil:.3f}')

# Visualisation PCA 2D
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
df_ml['pca1'] = coords[:, 0]
df_ml['pca2'] = coords[:, 1]

fig = px.scatter(
    df_ml, x='pca1', y='pca2',
    color=df_ml['cluster'].astype(str),
    title=f'Clustering K-Means (k={K_FINAL}) — Projection PCA 2D',
    labels={'pca1': 'Composante 1', 'pca2': 'Composante 2', 'color': 'Cluster'},
    opacity=0.5, height=450
)
fig.show()

In [ ]:
# Profil moyen de chaque cluster — base pour les nommer
profile = df_ml.groupby('cluster')[CLUSTER_FEATS].mean().round(2)
print('Profil moyen des clusters :')
print(profile.to_string())

# Ton moyen par cluster
if COL_TONE:
    tone_by_cluster = df_ml.groupby('cluster')[COL_TONE].mean().round(2)
    print('\nTon médiatique moyen par cluster :')
    print(tone_by_cluster)

print()
print('→ Nomme chaque cluster dans la cellule suivante en fonction du profil ci-dessus.')

In [ ]:
# !! ADAPTE CES NOMS SELON TES OBSERVATIONS !!
# Les 4 noms ci-dessous sont des suggestions — modifie selon les profils réels.
# Si K_FINAL != 4, ajoute ou supprime des lignes en conséquence.
CLUSTER_NAMES = {
    0: 'Coopération verbale',
    1: 'Coopération matérielle',
    2: 'Conflit verbal',
    3: 'Conflit matériel',
    # ajoute autant de lignes que de clusters
}

# Fallback automatique pour les clusters sans nom dans le dict
all_clusters = sorted(df_ml['cluster'].unique())
for k in all_clusters:
    if k not in CLUSTER_NAMES:
        CLUSTER_NAMES[k] = f'Cluster {k}'

df_ml['cluster_name'] = df_ml['cluster'].map(CLUSTER_NAMES)

print('Répartition par cluster :')
print(df_ml['cluster_name'].value_counts())

# Camembert des clusters
fig = px.pie(
    df_ml['cluster_name'].value_counts().reset_index(),
    names='cluster_name', values='count',
    title='Répartition des événements par profil (cluster)',
    hole=0.4
)
fig.show()


---
## 3. Classification du sentiment médiatique
**Objectif** : Prédire si la couverture d'un événement est Positive / Neutre / Négative.

In [ ]:
CLASSIF_FEATS = [c for c in [COL_GOLDSTEIN, COL_QUADCLASS, COL_MENTIONS,
                               COL_SOURCES, COL_ARTICLES,
                               '_month', '_dow', '_is_conflict', 'cluster'] if c in df_ml.columns]
TARGET = '_sentiment'

X = df_ml[CLASSIF_FEATS]
y = df_ml[TARGET]

# Retirer les nan restants
mask = X.notna().all(axis=1) & y.notna()
X, y = X[mask], y[mask]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train : {len(X_train):,} | Test : {len(X_test):,}')
print('Distribution classes :', y.value_counts().to_dict())

In [ ]:
# Entraîner Random Forest
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('=== RAPPORT DE CLASSIFICATION ===')
print(classification_report(y_test, y_pred))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred, labels=['Negatif', 'Neutre', 'Positif'])
fig = px.imshow(
    cm,
    x=['Negatif', 'Neutre', 'Positif'],
    y=['Negatif', 'Neutre', 'Positif'],
    color_continuous_scale='Blues',
    title='Matrice de confusion — Classifieur Sentiment',
    labels=dict(x='Prédit', y='Réel', color='Nombre'),
    text_auto=True, height=350
)
fig.show()

In [ ]:
# Validation croisée stratifiée
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
print(f'F1-macro (CV 5-fold) : {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print(f'Scores par fold      : {[round(s, 3) for s in cv_scores]}')

In [ ]:
# Importance des variables
importances = pd.DataFrame({
    'feature': CLASSIF_FEATS,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)

fig = px.bar(
    importances.sort_values('importance'),
    x='importance', y='feature', orientation='h',
    title='Importance des variables — Prédiction du sentiment médiatique',
    labels={'importance': 'Importance', 'feature': 'Variable'},
    color='importance', color_continuous_scale='Purples',
    height=400
)
fig.update_layout(showlegend=False)
fig.show()

print('\nTop 3 variables les plus prédictives :')
for _, row in importances.head(3).iterrows():
    print(f"  {row['feature']:30s} : {row['importance']:.3f}")

---
## 4. Comparaison Random Forest vs Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

from sklearn.metrics import f1_score
f1_rf = f1_score(y_test, y_pred, average='macro')
f1_gb = f1_score(y_test, y_pred_gb, average='macro')

comparison = pd.DataFrame({
    'Modele': ['Random Forest', 'Gradient Boosting'],
    'F1_macro_test': [f1_rf, f1_gb]
})
fig = px.bar(
    comparison, x='Modele', y='F1_macro_test',
    title='Comparaison des modèles — F1-macro sur le jeu de test',
    color='Modele', color_discrete_sequence=['#6c5ce7', '#e17055'],
    text_auto='.3f', height=350
)
fig.update_layout(showlegend=False, yaxis_range=[0, 1])
fig.show()

best_model = clf if f1_rf >= f1_gb else gb
best_name  = 'RandomForest' if f1_rf >= f1_gb else 'GradientBoosting'
print(f'\nMeilleur modele : {best_name} (F1-macro = {max(f1_rf, f1_gb):.3f})')

---
## 5. Sauvegarde des modèles

In [ ]:
joblib.dump(best_model, MODELS_DIR / 'sentiment_classifier.pkl')
joblib.dump(kmeans,     MODELS_DIR / 'kmeans_events.pkl')
joblib.dump(scaler,     MODELS_DIR / 'scaler_cluster.pkl')

# Sauvegarder les métadonnées
# Les clés int de CLUSTER_NAMES sont converties en str pour un JSON propre
import json as _json
cluster_names_str = {str(k): v for k, v in CLUSTER_NAMES.items()}
meta = {
    'best_model': best_name,
    'f1_macro_test': round(max(f1_rf, f1_gb), 4),
    'cv_f1_macro_mean': round(cv_scores.mean(), 4),
    'cv_f1_macro_std': round(cv_scores.std(), 4),
    'n_clusters': K_FINAL,
    'silhouette_score': round(sil, 4),
    'features': CLASSIF_FEATS,
    'cluster_names': cluster_names_str
}
(MODELS_DIR / 'model_metadata.json').write_text(_json.dumps(meta, indent=2, ensure_ascii=False))

print('Modeles et métadonnées sauvegardés dans /models/')
print(_json.dumps(meta, indent=2, ensure_ascii=False))


---
## 6. Résumé final pour le rapport

In [ ]:
print('=' * 65)
print('RESUME ML — A COPIER DANS LE RAPPORT')
print('=' * 65)
print(f'\nCLUSTERING')
print(f'  Algorithme     : K-Means (k={K_FINAL})')
print(f'  Score silhouette : {sil:.3f} (0=aléatoire, 1=parfait)')
print(f'  Profils identifies : {", ".join(CLUSTER_NAMES.values())}')
print(f'\nCLASSIFICATION')
print(f'  Meilleur modele  : {best_name}')
print(f'  F1-macro (test)  : {max(f1_rf, f1_gb):.3f}')
print(f'  F1-macro (CV 5x) : {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')
print(f'  Variable la + predictive : {importances.iloc[0]["feature"]}')